# Simple RNN Text Generation with TensorFlow

This notebook demonstrates text generation using a Recurrent Neural Network (RNN):

**Key Concepts:**
- **Tokenization**: Converting text to numerical sequences
- **Embedding Layer**: Learning word vector representations
- **SimpleRNN**: Processing sequential data with memory
- **Next Word Prediction**: Generating text word by word

In [ ]:
import tensorflow as tf
import numpy as np

print(f"TensorFlow version: {tf.__version__}")

## Step 1: Create Training Corpus

A small corpus of Indian patriotic phrases for demonstration.

In [ ]:
# Small training corpus
docs = ["go india", "india will win", "hip hip hooray", "bharat mata jai"]
print("Training corpus:")
for i, doc in enumerate(docs):
    print(f"  {i+1}. {doc}")

## Step 2: Tokenize Words

Convert text to numerical sequences using Keras Tokenizer.

In [ ]:
# Tokenize words
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(docs)
sequences = tokenizer.texts_to_sequences(docs)

print("Word Index (Vocabulary):")
for word, idx in tokenizer.word_index.items():
    print(f"  '{word}': {idx}")

print(f"\nTokenized sequences: {sequences}")

## Step 3: Prepare Input-Output Pairs

Create training pairs where X = sequence of words, y = next word to predict.

In [ ]:
# Prepare input-output pairs (predict next word)
X = []
y = []
for seq in sequences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

print("Training pairs (before padding):")
for i, (inp, out) in enumerate(zip(X, y)):
    print(f"  X: {inp} -> y: {out}")

In [ ]:
# Pad sequences to same length
X = tf.keras.preprocessing.sequence.pad_sequences(X, padding='pre')
y = np.array(y)

print(f"Padded X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nPadded X:\n{X}")

## Step 4: Build RNN Model

Architecture:
- **Embedding**: Converts word indices to dense vectors (10 dimensions)
- **SimpleRNN**: Processes sequences with 50 recurrent units
- **Dense**: Outputs probabilities over vocabulary (softmax)

In [ ]:
# Build RNN model
vocab_size = len(tokenizer.word_index) + 1  # +1 for padding token

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 10, input_length=X.shape[1]),
    tf.keras.layers.SimpleRNN(50),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Step 5: Train the Model

In [ ]:
# Train the model
print("Training for 200 epochs...")
history = model.fit(X, y, epochs=200, verbose=0)
print(f"Final loss: {history.history['loss'][-1]:.4f}")

## Step 6: Generate Text

Define a function to generate new text by predicting next words iteratively.

In [ ]:
def generate_text(seed_text, next_words=4):
    """Generate text by predicting next words."""
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([seed_text])[0]
        seq = tf.keras.preprocessing.sequence.pad_sequences([seq], maxlen=X.shape[1], padding='pre')
        pred = np.argmax(model.predict(seq, verbose=0))
        word = tokenizer.index_word[pred]
        seed_text += " " + word
    return seed_text

In [ ]:
# Generate text starting with "india"
result = generate_text("india")
print(f"Generated text: {result}")

In [ ]:
# Try more seed words
print("Text Generation Results:")
print("-" * 40)
for seed in ["go", "hip", "bharat"]:
    generated = generate_text(seed, next_words=3)
    print(f"Seed: '{seed}' -> '{generated}'")